# Data Preparation & Normalisation

**Dataset**: AG News (Text Classification)  
**Task**: Classify news articles into 4 categories: World, Sports, Business, Sci/Tech  
**Modality**: Text

In [1]:
import json
import os
import re
import string

import pandas as pd
import numpy as np
from datasets import load_dataset

/Users/yuvarajkosuru/Downloads/IIT_2/ML_ops/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Inspect Raw Data
Examine the dataset size, structure, class distribution, and any quality issues.

In [2]:
# Load AG News dataset from HuggingFace
print("Loading AG News dataset from HuggingFace...")
dataset = load_dataset("ag_news")

# Convert to pandas DataFrames
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

# Rename columns to match expected format
# HuggingFace ag_news has 'label' (0-3) and 'text' columns
train_df = train_df.rename(columns={"text": "Description"})
test_df = test_df.rename(columns={"text": "Description"})

# Add Title column (AG News from HF doesn't separate title/description)
# We'll extract first sentence as title for better text cleaning
train_df["Title"] = train_df["Description"].str.split('.').str[0]
test_df["Title"] = test_df["Description"].str.split('.').str[0]

# Rename 'label' to 'Class Index' and adjust to 1-based indexing (0-3 → 1-4)
train_df["Class Index"] = train_df["label"] + 1
test_df["Class Index"] = test_df["label"] + 1

# Drop the original 'label' column
train_df = train_df.drop(columns=["label"])
test_df = test_df.drop(columns=["label"])

print(f"Train set shape: {train_df.shape}")
print(f"Test set shape:  {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nData types:\n{train_df.dtypes}")

Loading AG News dataset from HuggingFace...
Train set shape: (120000, 3)
Test set shape:  (7600, 3)

Columns: ['Description', 'Title', 'Class Index']

Data types:
Description       str
Title          object
Class Index     int64
dtype: object


In [3]:
# Preview the data
train_df.head()

,Description,Title,Class Index
0,Wall St. Bears Claw Back Into the Black (Reute...,Wall St,3
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Carlyle Looks Toward Commercial Aerospace (Reu...,3
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,Oil and Economy Cloud Stocks' Outlook (Reuters...,3
3,Iraq Halts Oil Exports from Main Southern Pipe...,Iraq Halts Oil Exports from Main Southern Pipe...,3
4,"Oil prices soar to all-time record, posing new...","Oil prices soar to all-time record, posing new...",3


In [4]:
# Class distribution
print("=== Train Set Class Distribution ===")
print(train_df["Class Index"].value_counts().sort_index())
print(f"\nBalance ratio (min/max): {train_df['Class Index'].value_counts().min() / train_df['Class Index'].value_counts().max():.4f}")

print("\n=== Test Set Class Distribution ===")
print(test_df["Class Index"].value_counts().sort_index())
print(f"\nBalance ratio (min/max): {test_df['Class Index'].value_counts().min() / test_df['Class Index'].value_counts().max():.4f}")

=== Train Set Class Distribution ===
Class Index
1    30000
2    30000
3    30000
4    30000
Name: count, dtype: int64

Balance ratio (min/max): 1.0000

=== Test Set Class Distribution ===
Class Index
1    1900
2    1900
3    1900
4    1900
Name: count, dtype: int64

Balance ratio (min/max): 1.0000


In [5]:
# Quality issues check
print("=== Missing Values (Train) ===")
print(train_df.isnull().sum())

print(f"\n=== Duplicate Rows ===")
print(f"Train duplicates: {train_df.duplicated().sum()}")
print(f"Test duplicates:  {test_df.duplicated().sum()}")

print(f"\n=== Text Length Statistics (Description) ===")
desc_len = train_df["Description"].str.len()
print(f"Min: {desc_len.min()}, Max: {desc_len.max()}, Mean: {desc_len.mean():.1f}, Median: {desc_len.median():.1f}")

print(f"\n=== Empty Fields ===")
print(f"Empty descriptions (train): {train_df['Description'].isna().sum() + (train_df['Description'] == '').sum()}")
print(f"Empty titles (train): {train_df['Title'].isna().sum() + (train_df['Title'] == '').sum()}")

=== Missing Values (Train) ===
Description    0
Title          0
Class Index    0
dtype: int64

=== Duplicate Rows ===
Train duplicates: 0
Test duplicates:  0

=== Text Length Statistics (Description) ===
Min: 100, Max: 1012, Mean: 236.5, Median: 232.0

=== Empty Fields ===
Empty descriptions (train): 0
Empty titles (train): 8


## 2. Clean & Normalise Text Data

Text cleaning steps applied:
1. Lowercase all text
2. Remove HTML entities (`&lt;`, `#36;`, etc.)
3. Remove URLs
4. Strip punctuation
5. Remove duplicate rows
6. Drop rows with missing/empty text
7. Normalize whitespace
8. Combine title + description into a single `text` field

In [6]:
def clean_text(text):
    """Clean and normalise a text string for NLP tasks."""
    if pd.isna(text) or text is None:
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove HTML entities and tags
    text = re.sub(r'&lt;.*?&gt;', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&quot;', '"', text)
    text = re.sub(r'&#?\w+;', '', text)

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)

    # Remove backslashes
    text = text.replace('\\', ' ')

    # Replace #36; encoding with $
    text = re.sub(r'#36;', '$', text)

    # Strip punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [7]:
def prepare_dataset(df, split_name="train"):
    """Full cleaning pipeline for the text dataset."""
    print(f"--- Cleaning {split_name} data ---")
    original_len = len(df)

    # Remove duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  After removing duplicates: {len(df)} rows (removed {original_len - len(df)})")

    # Handle missing values
    df = df.dropna(subset=["Title", "Description"]).reset_index(drop=True)
    df = df[df["Description"].str.strip() != ""].reset_index(drop=True)
    df = df[df["Title"].str.strip() != ""].reset_index(drop=True)
    print(f"  After removing missing/empty text: {len(df)} rows")

    # Clean text columns
    df["Title_clean"] = df["Title"].apply(clean_text)
    df["Description_clean"] = df["Description"].apply(clean_text)

    # Combine title and description
    df["text"] = df["Title_clean"] + " " + df["Description_clean"]
    df["text"] = df["text"].str.strip()

    # Remove rows empty after cleaning
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    print(f"  After text cleaning: {len(df)} rows")

    # Encode labels (0-indexed)
    df["label"] = df["Class Index"] - 1

    print(f"  Final dataset: {len(df)} rows")
    return df

In [8]:
# Apply cleaning pipeline
train_clean = prepare_dataset(train_df, "train")
test_clean = prepare_dataset(test_df, "test")

--- Cleaning train data ---
  After removing duplicates: 120000 rows (removed 0)
  After removing missing/empty text: 119992 rows
  After text cleaning: 119992 rows
  Final dataset: 119992 rows
--- Cleaning test data ---
  After removing duplicates: 7600 rows (removed 0)
  After removing missing/empty text: 7599 rows
  After text cleaning: 7599 rows
  Final dataset: 7599 rows


In [9]:
# Show sample of cleaned text
print("Sample cleaned texts:")
for i in range(3):
    print(f"\n[Label {train_clean.iloc[i]['label']}] {train_clean.iloc[i]['text'][:150]}...")

Sample cleaned texts:

[Label 2] wall st wall st bears claw back into the black reuters reuters shortsellers wall streets dwindling band of ultracynics are seeing green again...

[Label 2] carlyle looks toward commercial aerospace reuters reuters private investment firm carlyle group which has a reputation for making welltimed and occasi...

[Label 2] oil and economy cloud stocks outlook reuters reuters soaring crude prices plus worries about the economy and the outlook for earnings are expected to ...


## 3. Encode Labels & Save Mapping

Create a `id2label.json` mapping file that maps integer label IDs to human-readable class names.

In [10]:
# AG News label mapping (0-indexed)
id2label = {
    "0": "World",
    "1": "Sports",
    "2": "Business",
    "3": "Sci/Tech"
}

# Save mapping file
with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

print("Saved id2label.json:")
print(json.dumps(id2label, indent=2))

Saved id2label.json:
{
  "0": "World",
  "1": "Sports",
  "2": "Business",
  "3": "Sci/Tech"
}


## 4. Save Prepared Dataset to Hugging Face

Push the cleaned dataset to Hugging Face Hub so it can be loaded directly in the Kaggle training notebook.

In [12]:
from datasets import Dataset, DatasetDict

# ⚠️ IMPORTANT: Change this to YOUR HuggingFace username!
# Get your username from: https://huggingface.co/settings/account
HF_USERNAME = "YOUR_HF_USERNAME"  # Replace with your actual username
HF_DATASET_REPO = f"{HF_USERNAME}/ag-news-prepared"

# Prepare final dataframes
train_final = train_clean[["text", "label"]].copy()
test_final = test_clean[["text", "label"]].copy()

# Check class distribution
print("Class distribution:")
print(train_final["label"].value_counts().sort_index())
print(f"\nUsing full cleaned dataset: {len(train_final)} train, {len(test_final)} test samples")

# Create DatasetDict
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_final, preserve_index=False),
    "test": Dataset.from_pandas(test_final, preserve_index=False),
})

print("\nDataset structure:")
print(dataset_dict)

# Optional: Push to HuggingFace Hub (requires HF token with write permissions)
# If you don't want to push to HF, you can skip this and load data in Kaggle differently
push_to_hf = False  # Set to True to push

if push_to_hf:
    if HF_USERNAME == "YOUR_HF_USERNAME":
        print("\n⚠️ ERROR: Please set your actual HuggingFace username first!")
        print("Get it from: https://huggingface.co/settings/account")
    else:
        from huggingface_hub import login
        # Login with your HF token (will prompt or use HF_TOKEN env variable)
        login()
        
        # Push to Hub
        dataset_dict.push_to_hub(HF_DATASET_REPO)
        print(f"\n✅ Dataset pushed successfully!")
        print(f"URL: https://huggingface.co/datasets/{HF_DATASET_REPO}")
else:
    print("\n⚠️ Skipping HuggingFace Hub push (set push_to_hf=True to enable)")
    print("Alternative: You can use the cleaned dataframes directly in Kaggle or save locally")
    
    # Save locally as CSV for Kaggle upload
    train_final.to_csv("train_cleaned.csv", index=False)
    test_final.to_csv("test_cleaned.csv", index=False)
    print("\n✅ Saved cleaned data locally:")
    print("  - train_cleaned.csv")
    print("  - test_cleaned.csv")
    print("\nYou can upload these to Kaggle as a dataset.")

Class distribution:
label
0    29999
1    30000
2    29999
3    29994
Name: count, dtype: int64

Using full cleaned dataset: 119992 train, 7599 test samples

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 119992
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7599
    })
})

⚠️ Skipping HuggingFace Hub push (set push_to_hf=True to enable)
Alternative: You can use the cleaned dataframes directly in Kaggle or save locally

✅ Saved cleaned data locally:
  - train_cleaned.csv
  - test_cleaned.csv

You can upload these to Kaggle as a dataset.


In [13]:
# Final summary
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Task: Text Classification (AG News)")
print(f"Classes: {len(id2label)} — {list(id2label.values())}")
print(f"Train samples: {len(train_final)}")
print(f"Test samples:  {len(test_final)}")
print(f"\nLabel distribution (train):")
for label_id, label_name in id2label.items():
    count = (train_final["label"] == int(label_id)).sum()
    print(f"  {label_id} ({label_name}): {count}")
print(f"\nDataset on HF Hub: https://huggingface.co/datasets/{HF_DATASET_REPO}")
print(f"id2label.json: saved locally (commit to git)")

SUMMARY
Task: Text Classification (AG News)
Classes: 4 — ['World', 'Sports', 'Business', 'Sci/Tech']
Train samples: 119992
Test samples:  7599

Label distribution (train):
  0 (World): 29999
  1 (Sports): 30000
  2 (Business): 29999
  3 (Sci/Tech): 29994

Dataset on HF Hub: https://huggingface.co/datasets/YOUR_HF_USERNAME/ag-news-prepared
id2label.json: saved locally (commit to git)
